# Lancaster County election results scraper

This is a Python Notebook built to scrape the Lancaster County election results API. It riffs off [https://palewi.re/docs/first-github-scraper/](https://palewi.re/docs/first-github-scraper/), because it's a great resource for troubleshooting or creating more scraper Notebooks as needs dictate.

### Notebook Steps

<details>
<summary>Table of Contents (click to expand)</summary>

1. [Env Setup](#env-setup)
1. [Import Python Libraries](#import-python-libraries)
1. [Check Lancaster County Elections API](#check-lancaster-county-elections-api)
1. [Define API Endpoints](#define-api-endpoints)
1. [Site-Change Detection](#site-change-detection)
1. [Get Lancaster County's list of all Elections](#get-lancaster-countys-list-of-all-elections)
1. [Print out all known elections](#print-out-all-known-elections)
1. [Get a Specific Election's CSV](#get-a-specific-elections-csv)
1. [Lookup Election Details from List](#lookup-election-details-from-list)
1. [Fetch election results data](#fetch-election-results-data)
1. [Export election data to CSV](#export-election-data-to-csv)
</details>

### Env Setup

In [ ]:
# Setup env: imports and dependencies for the project
import os # for working with the operating system, like file paths
import sys # for checking system information
from pathlib import Path # for working with file paths

# add current directory to sys.path so we can import shared_setup
sys.path.insert(0, str(Path.cwd())) 

# import a shared tool to check dependencies
from shared_setup import check_dependencies
check_dependencies()

# icons
iWarn = "⚠️"
iOK = "✅"
iErr = "❌"

### Import Python Libraries

In [ ]:
import requests # for making HTTP requests
import csv # for working with CSV files
import json # for working with JSON data
import time # for adding delays between requests
from datetime import datetime # for working with dates

# Create a session to maintain cookies across requests
session = requests.Session()

# Set up realistic browser headers
session.headers.update({
    'User-Agent': 'PublicLedger newsgathering info@publicledger.news Mozilla/5.0',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'DNT': '1',
    'Connection': 'keep-alive',
    'Cache-Control': 'no-cache',
    'Pragma': 'no-cache'
})

### Check Lancaster County Elections API

[https://electionresults.lancastercountypa.gov/](https://electionresults.lancastercountypa.gov/)

Lancaster County provides a public API for accessing election results data.

#### Define API Endpoints

Let's break down the Lancaster County API into its main components.

In [ ]:
# Lancaster County API base URL and endpoints
lancasterSite = 'https://electionresults.lancastercountypa.gov/'
lancasterApiBase = lancasterSite + 'results/public/api/'
lancasterJurisdiction = lancasterApiBase + 'jurisdictions/lancaster-county-pa'
lancasterElections = lancasterApiBase + 'elections'

#### Site-Change Detection

First, let's verify we can access the jurisdiction endpoint and check the response structure to confirm the API has not changed.

In [ ]:
# First, hit the jurisdiction endpoint to establish cookies/session
jurisdictionRes = session.get(lancasterJurisdiction, timeout=30)
# Check the Content-Encoding header to determine decompression method
content_encoding = jurisdictionRes.headers.get('Content-Encoding', '').lower()

compression = ''
if content_encoding == 'br':
    import brotli
    decompressed = brotli.decompress(jurisdictionRes.content)
    response_text = decompressed.decode('utf-8')
elif content_encoding == 'gzip' or jurisdictionRes.content[:2] == b'\x1f\x8b':
    import gzip
    decompressed = gzip.decompress(jurisdictionRes.content)
    response_text = decompressed.decode('utf-8')
else:
    response_text = jurisdictionRes.text

trim = 100
summary = [
    f"Response Status: {jurisdictionRes.status_code} | Content-Encoding: {jurisdictionRes.headers.get('Content-Encoding', 'not specified')}",
    f"First {trim} characters of response: {response_text[:trim]}"
]
print("\n".join(summary))


# Only raise for serious HTTP errors (4xx, 5xx)
if jurisdictionRes.status_code >= 400:
    jurisdictionRes.raise_for_status()

# Parse JSON response
try:
    jurisdictionData = json.loads(response_text)
    
    # Display basic info about the jurisdiction
    if isinstance(jurisdictionData, dict):
        # print(f"JSON Keys in response: {list(jurisdictionData.keys())}")
        
        # Extract jurisdiction name (it's in a multi-language array)
        if 'name' in jurisdictionData and isinstance(jurisdictionData['name'], list):
            jurisdiction_name = jurisdictionData['name'][0].get('text', 'Unknown')
            print(f"{iOK} Jurisdiction: {jurisdiction_name}")
        
        # Check if elections list is present
        if 'elections' in jurisdictionData:
            print(f"{iOK} Found 'elections' key with {len(jurisdictionData['elections'])} elections")
        else:
            print(f"{iWarn} No 'elections' key found in response")
            
    else:
        print(f"{iErr} Unexpected response format: {type(jurisdictionData)}")
        print(f"{iWarn} API structure may have changed")
        exit(1)
        
except json.JSONDecodeError as e:
    print(f"\n{iErr} Failed to parse JSON response")
    print(f"Error: {e}")
    print(f"\n{iWarn} The endpoint is not returning valid JSON")
    print(f"Check the response content above to see what was actually returned.")
    exit(1)

## Get Lancaster County's list of all Elections

This next element will fetch the list of elections from Lancaster County and display them.

In [ ]:
# Extract the elections list from the jurisdiction data
# The jurisdiction endpoint already gave us all elections, so we just extract them
print("Extracting elections list from jurisdiction data...")

if 'elections' in jurisdictionData and isinstance(jurisdictionData['elections'], list):
    electionsList = jurisdictionData['elections']
    print(f"{iOK} Found {len(electionsList)} elections")
    
    if electionsList:
        # Show the structure of the first election
        print(f"\nSample election record keys: {list(electionsList[0].keys())}")
else:
    print(f"{iErr} Could not extract elections from jurisdiction data")
    print(f"Available keys: {list(jurisdictionData.keys())}")
    electionsList = []

#### Print out all known elections

In [ ]:
import glob
from shared_table_display import display_markdown_table, sanitize_markdown_cell

# Display elections in a formatted table
if electionsList and len(electionsList) > 0:
    # Define columns to display
    columns = ['Collected', 'publicElectionId', 'name', 'electionDate', 'dateText']
    
    def extract_text(lang_array):
        """Extract English text from multi-language array."""
        if isinstance(lang_array, list) and len(lang_array) > 0:
            return lang_array[0].get('text', '')
        return ''
    
    def check_collected(row):
        """Return 🟢 if CSV exists, ⚫ if no data placeholder exists, ❌ if not collected."""
        try:
            date_str = row.get('electionDate', '')
            if not date_str:
                return '❌'
            
            # Parse ISO date format (YYYY-MM-DD)
            date_obj = datetime.strptime(date_str, '%Y-%m-%d')
            formatted_date = date_obj.strftime("%Y-%m-%d")
            year = date_obj.year
        except (ValueError, AttributeError):
            return '❌'

        election_id = row.get('publicElectionId', '')
        # Lancaster County doesn't seem to have a type field, use '*' as wildcard
        election_type = '*'

        base_dir = os.path.join(
            '..', 'data', 'raw_notebook_csvs', 'county',
            'Lancaster', str(year), str(election_type),
            str(formatted_date)
        )
        
        # Check for both regular and x_ prefixed files
        pattern = os.path.join(base_dir, f"{election_id}-*.csv")
        x_pattern = os.path.join(base_dir, f"x_{election_id}-*.csv")
        
        matches = glob.glob(pattern)
        x_matches = glob.glob(x_pattern)
        
        if x_matches:
            return '⚫'
        elif matches:
            return '🟢'
        else:
            return '❌'
    
    # Prepare rows with extracted text and collected status
    rows_to_display = []
    for row in electionsList:
        display_row = {
            'Collected': check_collected(row),
            'publicElectionId': row.get('publicElectionId', ''),
            'name': extract_text(row.get('name', [])),
            'electionDate': row.get('electionDate', ''),
            'dateText': extract_text(row.get('dateText', []))
        }
        rows_to_display.append(display_row)
    
    # Sort by date (most recent first)
    def sort_key(row):
        try:
            return datetime.strptime(row.get('electionDate', ''), '%Y-%m-%d')
        except:
            return datetime.min
    
    rows_to_display = sorted(rows_to_display, key=sort_key, reverse=True)
    
    # Display the table
    markdown_table = display_markdown_table(rows_to_display, columns)
    
    # Export table to data/raw_notebook_csvs/county/Lancaster/COUNTY_README.md
    export_path = Path('..') / 'data' / 'raw_notebook_csvs' / 'county' / 'Lancaster'
    export_path.mkdir(parents=True, exist_ok=True)
    readme_path = export_path / 'COUNTY_README.md'
    
    with open(readme_path, 'w', encoding='utf-8') as f:
        f.write(f"# Lancaster County Elections\n\n")
        f.write(f"Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write(markdown_table)
    
    print(f"\n{iOK} Exported table to {readme_path}")
else:
    print(f"{iWarn} No elections found to display")

### Get a Specific Election's Data

We'll use the list of elections we already gathered to populate some details their service needs.

In [ ]:
electionKey = "2024GeneralPrimary" # slug chosen from list above

### Prepare to Fetch Election Details from List

In [ ]:
# Lookup election details from the elections list we extracted earlier
if electionKey is None:
    print(f"{iWarn} Please set electionKey in the cell above first")
    exit(1)

# Find the election in our list by publicElectionId
selectedElection = None
for election in electionsList:
    if election.get('publicElectionId') == electionKey:
        selectedElection = election
        break

# Throw an error if we couldn't find the election
if selectedElection is None:
    raise ValueError(f'Could not find election with publicElectionId: {electionKey}')

# Extract election details (names and dates are in multi-language arrays)
publicElectionId = selectedElection.get('publicElectionId', '')
elexDate = selectedElection.get('electionDate', '')
elexName = selectedElection['name'][0].get('text', f'Election {publicElectionId}') if 'name' in selectedElection else f'Election {publicElectionId}'
elexDateText = selectedElection['dateText'][0].get('text', elexDate) if 'dateText' in selectedElection else elexDate

print(f"{iOK} Selected election: {elexName}")
print(f"  Public ID: {publicElectionId}")
print(f"  Date: {elexDateText}")

### Fetch election results data

Request the full election data from the correct API endpoint.

In [ ]:
# Construct the correct URL for election data
# Pattern: api/elections/lancaster-county-pa/{publicElectionId}/data
electionDataUrl = lancasterApiBase + f'elections/lancaster-county-pa/{publicElectionId}/data'

print(f"Fetching election data from:\n{electionDataUrl}\n")

# Update headers to match working curl
session.headers.update({
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Cache-Control': 'no-cache',
    'DNT': '1',
})

# Fetch the election data
dataRes = session.get(electionDataUrl, timeout=30)
print(f"Status: {dataRes.status_code}")

if dataRes.status_code == 200:
    # Handle potential Brotli compression
    content_encoding = dataRes.headers.get('Content-Encoding', '').lower()
    if content_encoding == 'br':
        import brotli
        response_text = brotli.decompress(dataRes.content).decode('utf-8')
    else:
        response_text = dataRes.text
    
    electionData = json.loads(response_text)
    print(f"{iOK} Successfully fetched election data")
    print(f"Data type: {type(electionData)}")
    
    if isinstance(electionData, dict):
        print(f"Keys: {list(electionData.keys())}")
    elif isinstance(electionData, list):
        print(f"Array length: {len(electionData)}")
else:
    print(f"{iWarn} Request failed with status {dataRes.status_code}")
    electionData = None

### Export election data to CSV

Extract the relevant data arrays from the election object and save as CSV.

In [ ]:
# Parse the date string and format it as YYYY-MM-DD for filename
try:
    # Date is already in YYYY-MM-DD format
    date_obj = datetime.strptime(elexDate, '%Y-%m-%d')
    formatted_date = date_obj.strftime("%Y-%m-%d")
    elexYear = date_obj.year
except (ValueError, AttributeError) as e:
    print(f"{iWarn} Error parsing date: {e}")
    # Use current date as fallback
    formatted_date = datetime.now().strftime("%Y-%m-%d")
    elexYear = datetime.now().year

# Infer ElectionType from the election name or publicElectionId
# Use same convention as PA state scraper: G=General, P=Primary, S=Special, O=Other
electionType_inference = {
    'primary': 'P',
    'general': 'G',
    'special': 'S',
}

# Check both name and publicElectionId for keywords
search_text = f"{elexName} {publicElectionId}".lower()
electionType = 'O'  # default to Other
for keyword, type_code in electionType_inference.items():
    if keyword in search_text:
        electionType = type_code
        break

print(f"{iOK} Inferred ElectionType: '{electionType}' from '{elexName}'")

# "slugify" the election name to make it safe for filenames
elexNameSlug = elexName.lower().replace(' ', '-').replace('/', '-')

# Add a marker for "now", the point of running the scraper
nowish = datetime.now().isoformat(timespec="seconds")

# Find the data array to export - look for common result keys
data_to_export = None
export_key = None
for key in ['contests', 'ballotItems', 'precincts', 'results', 'races', 'candidates']:
    if key in electionData and isinstance(electionData[key], list) and len(electionData[key]) > 0:
        data_to_export = electionData[key]
        export_key = key
        print(f"{iOK} Found data to export: '{key}' with {len(data_to_export)} records")
        break

if data_to_export is None:
    print(f"{iWarn} No recognizable data arrays found in election object")
    print(f"Available keys: {list(electionData.keys())}")
    print(f"Will create a placeholder file")

# Base directory for this election
saveDir = os.path.join(
    '..', 
    'data', 
    'raw_notebook_csvs', 
    'county',
    'Lancaster', 
    str(elexYear), 
    electionType,  # Inferred from election name
    str(formatted_date),
)

# Create directory if it doesn't exist
os.makedirs(saveDir, exist_ok=True)

hasData = data_to_export is not None and len(data_to_export) > 0

# Use x_ prefix for files with no data
prefix = "" if hasData else "x_"
baseFilename = f"{prefix}{publicElectionId}-{elexNameSlug}.csv"

# If the base filename already exists, append a datestamp to avoid overwriting it
if os.path.exists(os.path.join(saveDir, baseFilename)):
    filename = f"{prefix}{publicElectionId}-{elexNameSlug}-{nowish}.csv"
else:
    filename = baseFilename

filePath = os.path.join(saveDir, filename)

# Save results to a CSV file
with open(filePath, 'w', newline='', encoding='utf-8') as outfile:
    writer = csv.writer(outfile)

    # Add comments at the top of the CSV with election info
    comments = [
        f"Election: {elexName} ({elexDateText}) - Public ID: {publicElectionId} - Lancaster County",
        f"Report generated on: {nowish}",
    ]
    if hasData:
        comments.append(f"Data source: electionData['{export_key}'] - {len(data_to_export)} records")
    else:
        comments.append(f"NOTE: No data arrays found - placeholder file only")
    
    for comment in comments:
        outfile.write(f"# {comment}\n")

    # Write data rows only if we have data
    if hasData:
        # Check if records are dicts (typical case)
        if isinstance(data_to_export[0], dict):
            # Extract all possible headers from all records (in case structure varies)
            all_keys = set()
            for record in data_to_export:
                if isinstance(record, dict):
                    all_keys.update(record.keys())
            
            headers = sorted(list(all_keys))
            writer.writerow(headers)
            
            # Write data rows
            for record in data_to_export:
                if isinstance(record, dict):
                    row = []
                    for h in headers:
                        value = record.get(h, '')
                        # Handle nested structures
                        if isinstance(value, (dict, list)):
                            row.append(json.dumps(value))
                        else:
                            row.append(value)
                    writer.writerow(row)
        else:
            # If not dicts, just write as simple rows
            for item in data_to_export:
                writer.writerow([item])

status = iOK if hasData else iWarn
print(f"{status} Saved {'data' if hasData else 'empty placeholder'} to {filePath}")
